# DPO-SDXL × DrawBench — Z-Sampling (Colab)

Generates Z-Sampling images with the DPO model

In [1]:
!pip install -q diffusers transformers accelerate safetensors

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import gc
import json
import os
import shutil
import tempfile
import time
from dataclasses import dataclass
from pathlib import Path

import torch
from diffusers import DDIMInverseScheduler, DDIMScheduler, StableDiffusionXLPipeline, UNet2DConditionModel
from diffusers.pipelines.stable_diffusion_xl.pipeline_output import StableDiffusionXLPipelineOutput
from diffusers.pipelines.stable_diffusion_xl.pipeline_stable_diffusion_xl import rescale_noise_cfg
from IPython.display import display
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"torch: {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Device: cuda
torch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB


## Config

In [14]:
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
DPO_UNET_ID = "mhdang/dpo-sdxl-text2image-v1"
HEIGHT = 1024
WIDTH = 1024
SEEDS = [2026] #[42, 123, 2026]

# Z-Sampling hyperparameters (from the paper's SDXL config)
GAMMA_1 = 5.5        # guidance scale for denoising (forward)
GAMMA_2 = 0.0        # guidance scale for inversion (backward)
INFER_STEP = 50      # total DDIM steps
LAMBDA_STEP = None   # how many early steps get zigzag (None = infer_step - 1)
T_MAX = 1            # number of zig-zag rounds per step

OUTPUT_DIR = Path("/content/outputs/dpo_sdxl_drawbench_zsampling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_DIR = Path("/content/drive/MyDrive/generative-modelling/dpo_sdxl_drawbench_zsampling")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_EVERY = 15

# How many prompts to run. Set to None for all 200.
NUM_PROMPTS = None
PROMPT_SEED = 42

## DrawBench Prompts (200)

In [15]:
import random
import pandas as pd

CSV_URL = "https://huggingface.co/datasets/sayakpaul/drawbench/raw/main/DrawBench%20Prompts%20-%20Sheet1.csv"
df = pd.read_csv(CSV_URL)
DRAWBENCH_PROMPTS = list(enumerate(df["Prompts"].tolist()))

if NUM_PROMPTS is not None and NUM_PROMPTS < len(DRAWBENCH_PROMPTS):
    rng = random.Random(PROMPT_SEED)
    selected_prompts = sorted(rng.sample(DRAWBENCH_PROMPTS, NUM_PROMPTS))
    print(f"Randomly selected {NUM_PROMPTS}/{len(DRAWBENCH_PROMPTS)} prompts (seed={PROMPT_SEED})")
else:
    selected_prompts = DRAWBENCH_PROMPTS
    print(f"Running all {len(selected_prompts)} prompts")

print(f"First 5: {[f'{i}: {p[:40]}...' for i, p in selected_prompts[:5]]}")
df.head()

Running all 200 prompts
First 5: ['0: A red colored car....', '1: A black colored car....', '2: A pink colored car....', '3: A black colored dog....', '4: A red colored dog....']


,Prompts,Category
0,A red colored car.,Colors
1,A black colored car.,Colors
2,A pink colored car.,Colors
3,A black colored dog.,Colors
4,A red colored dog.,Colors


In [16]:
selected_prompts = selected_prompts[45:]

## Load Pipeline (DPO UNet)

In [8]:
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, use_safetensors=True
)

# Replace UNet with DPO-finetuned version
unet = UNet2DConditionModel.from_pretrained(
    DPO_UNET_ID, subfolder="unet", torch_dtype=DTYPE
)
pipe.unet = unet

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
inverse_scheduler = DDIMInverseScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)

if DEVICE == "cuda":
    pipe.enable_attention_slicing()
    pipe.enable_vae_tiling()

print(f"Pipeline loaded with DPO UNet from {DPO_UNET_ID}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/1.77k [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/10.3G [00:00<?, ?B/s]

The config attributes {'interpolation_type': 'linear', 'use_karras_sigmas': False, 'skip_prk_steps': True} were passed to DDIMInverseScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.


Pipeline loaded with DPO UNet from mhdang/dpo-sdxl-text2image-v1


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2294: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `StableDiffusionXLPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(


## Z-Sampling Implementation

Duplicated from `src/zigzag_project/zsampling.py` to avoid local package install

In [17]:
@dataclass(frozen=True)
class ZSamplingConfig:
    gamma_1: float = 5.5
    gamma_2: float = 0.0
    infer_step: int = 50
    lambda_step: int | None = None
    t_max: int = 1

    def resolved_lambda_step(self) -> int:
        if self.lambda_step is None:
            return self.infer_step - 1
        return self.lambda_step


def _guided_noise_prediction(
    pipe, latents, timestep, prompt_embeds, add_text_embeds, add_time_ids,
    guidance_scale, guidance_rescale, timestep_cond,
):
    latent_model_input = torch.cat([latents] * 2) if pipe.do_classifier_free_guidance else latents
    latent_model_input = pipe.scheduler.scale_model_input(latent_model_input, timestep)
    noise_pred = pipe.unet(
        latent_model_input, timestep,
        encoder_hidden_states=prompt_embeds,
        timestep_cond=timestep_cond,
        cross_attention_kwargs=pipe.cross_attention_kwargs,
        added_cond_kwargs={"text_embeds": add_text_embeds, "time_ids": add_time_ids},
        return_dict=False,
    )[0]
    if pipe.do_classifier_free_guidance:
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
        if guidance_rescale > 0.0:
            noise_pred = rescale_noise_cfg(noise_pred, noise_pred_text, guidance_rescale=guidance_rescale)
    return noise_pred


def _decode_latents(pipe, latents, output_type):
    if output_type == "latent":
        return latents
    needs_upcasting = pipe.vae.dtype == torch.float16 and pipe.vae.config.force_upcast
    if needs_upcasting:
        pipe.upcast_vae()
        latents = latents.to(next(iter(pipe.vae.post_quant_conv.parameters())).dtype)
    has_latents_mean = hasattr(pipe.vae.config, "latents_mean") and pipe.vae.config.latents_mean is not None
    has_latents_std = hasattr(pipe.vae.config, "latents_std") and pipe.vae.config.latents_std is not None
    if has_latents_mean and has_latents_std:
        latents_mean = torch.tensor(pipe.vae.config.latents_mean).view(1, 4, 1, 1).to(latents.device, latents.dtype)
        latents_std = torch.tensor(pipe.vae.config.latents_std).view(1, 4, 1, 1).to(latents.device, latents.dtype)
        latents = latents * latents_std / pipe.vae.config.scaling_factor + latents_mean
    else:
        latents = latents / pipe.vae.config.scaling_factor
    image = pipe.vae.decode(latents, return_dict=False)[0]
    if needs_upcasting:
        pipe.vae.to(dtype=torch.float16)
    if pipe.watermark is not None:
        image = pipe.watermark.apply_watermark(image)
    return pipe.image_processor.postprocess(image, output_type=output_type)


@torch.no_grad()
def z_sample_sdxl(
    pipe, inverse_scheduler, prompt, *,
    config=None, height=1024, width=1024,
    negative_prompt=None, generator=None, latents=None,
    output_type="pil", guidance_rescale=0.0,
):
    """Run SDXL Z-Sampling. Standalone version for Colab (no ModelHandle needed)."""
    config = config or ZSamplingConfig()
    lambda_step = config.resolved_lambda_step()

    original_size = (height, width)
    target_size = (height, width)
    crops_coords_top_left = (0, 0)

    pipe._guidance_scale = config.gamma_1
    pipe._guidance_rescale = guidance_rescale
    pipe._clip_skip = None
    pipe._cross_attention_kwargs = None
    pipe._denoising_end = None
    pipe._interrupt = False

    batch_size = 1 if isinstance(prompt, str) else len(prompt)
    device = pipe._execution_device

    (prompt_embeds, negative_prompt_embeds,
     pooled_prompt_embeds, negative_pooled_prompt_embeds) = pipe.encode_prompt(
        prompt=prompt, prompt_2=None, device=device,
        num_images_per_prompt=1,
        do_classifier_free_guidance=pipe.do_classifier_free_guidance,
        negative_prompt=negative_prompt,
    )

    pipe.scheduler.set_timesteps(config.infer_step, device=device)
    timesteps = pipe.scheduler.timesteps
    num_inference_steps = len(timesteps)
    inverse_scheduler.set_timesteps(num_inference_steps, device=device)

    latents = pipe.prepare_latents(
        batch_size, pipe.unet.config.in_channels,
        height, width, prompt_embeds.dtype, device, generator, latents,
    )
    extra_step_kwargs = pipe.prepare_extra_step_kwargs(generator, 0.0)

    add_text_embeds = pooled_prompt_embeds
    if pipe.text_encoder_2 is None:
        text_encoder_projection_dim = int(pooled_prompt_embeds.shape[-1])
    else:
        text_encoder_projection_dim = pipe.text_encoder_2.config.projection_dim

    add_time_ids = pipe._get_add_time_ids(
        original_size, crops_coords_top_left, target_size,
        dtype=prompt_embeds.dtype,
        text_encoder_projection_dim=text_encoder_projection_dim,
    )
    negative_add_time_ids = add_time_ids

    if pipe.do_classifier_free_guidance:
        prompt_embeds = torch.cat([negative_prompt_embeds, prompt_embeds], dim=0)
        add_text_embeds = torch.cat([negative_pooled_prompt_embeds, add_text_embeds], dim=0)
        add_time_ids = torch.cat([negative_add_time_ids, add_time_ids], dim=0)

    prompt_embeds = prompt_embeds.to(device)
    add_text_embeds = add_text_embeds.to(device)
    add_time_ids = add_time_ids.to(device).repeat(batch_size, 1)

    timestep_cond = None

    for i, timestep in enumerate(timesteps):
        noise_pred = _guided_noise_prediction(
            pipe, latents, timestep, prompt_embeds,
            add_text_embeds, add_time_ids, config.gamma_1, guidance_rescale, timestep_cond,
        )
        pipe.scheduler._step_index = None
        next_latents = pipe.scheduler.step(
            noise_pred, timestep, latents, **extra_step_kwargs, return_dict=False,
        )[0]

        if i < lambda_step:
            for _ in range(config.t_max):
                # Zag: inversion with weak guidance
                inv_noise_pred = _guided_noise_prediction(
                    pipe, next_latents, timestep, prompt_embeds,
                    add_text_embeds, add_time_ids, config.gamma_2, guidance_rescale, timestep_cond,
                )
                inverse_scheduler._step_index = None
                inv_latents = inverse_scheduler.step(
                    inv_noise_pred, timesteps[i + 1], next_latents, return_dict=False,
                )[0]

                # Zig: forward denoising with strong guidance
                fwd_noise_pred = _guided_noise_prediction(
                    pipe, inv_latents, timestep, prompt_embeds,
                    add_text_embeds, add_time_ids, config.gamma_1, guidance_rescale, timestep_cond,
                )
                pipe.scheduler._step_index = None
                next_latents = pipe.scheduler.step(
                    fwd_noise_pred, timestep, inv_latents, **extra_step_kwargs, return_dict=False,
                )[0]

        latents = next_latents

    image = _decode_latents(pipe, latents, output_type)
    pipe.maybe_free_model_hooks()
    return StableDiffusionXLPipelineOutput(images=image)

## Generate

In [18]:
zs_config = ZSamplingConfig(
    gamma_1=GAMMA_1,
    gamma_2=GAMMA_2,
    infer_step=INFER_STEP,
    lambda_step=LAMBDA_STEP,
    t_max=T_MAX,
)
print(f"Z-Sampling config: {zs_config}")
print(f"Effective lambda_step: {zs_config.resolved_lambda_step()}")
print(f"UNet evals per step (first {zs_config.resolved_lambda_step()} steps): {1 + 2 * T_MAX}")

Z-Sampling config: ZSamplingConfig(gamma_1=5.5, gamma_2=0.0, infer_step=50, lambda_step=None, t_max=1)
Effective lambda_step: 49
UNet evals per step (first 49 steps): 3


In [19]:
def checkpoint_to_drive(files, tag):
    """Zip only the given files and copy to Drive with flush."""
    with tempfile.TemporaryDirectory() as tmp:
        for f in files:
            shutil.copy2(f, Path(tmp) / f.name)
        zp = shutil.make_archive(f"/content/{tag}", "zip", tmp)
    dest = DRIVE_DIR / f"{tag}.zip"
    shutil.copy2(zp, dest)
    os.sync()
    size_mb = Path(zp).stat().st_size / 1e6
    print(f"  >> Checkpoint saved: {dest.name} ({size_mb:.1f} MB, {len(files)} images)")

total = len(SEEDS) * len(selected_prompts)
done = 0

for seed in SEEDS:
    seed_dir = OUTPUT_DIR / f"seed{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    manifest = seed_dir / "manifest.jsonl"
    pending_files = []
    ckpt_count = 3

    for prompt_idx, prompt in selected_prompts:
        t0 = time.time()
        generator = torch.Generator(device=DEVICE).manual_seed(seed)
        result = z_sample_sdxl(
            pipe, inverse_scheduler, prompt,
            config=zs_config,
            height=HEIGHT, width=WIDTH,
            generator=generator,
        )
        img = result.images[0]
        fname = f"dpo_sdxl_zsample_prompt{prompt_idx:03d}_seed{seed}.png"
        path = seed_dir / fname
        img.save(path)
        pending_files.append(path)
        del result, img
        torch.cuda.empty_cache()

        with open(manifest, "a") as f:
            f.write(json.dumps({"prompt_id": prompt_idx, "prompt": prompt, "seed": seed, "file": fname}) + "\n")

        done += 1
        elapsed = time.time() - t0
        print(f"[{done}/{total}] seed={seed} prompt={prompt_idx} ({elapsed:.1f}s)")

        if len(pending_files) == CHECKPOINT_EVERY:
            ckpt_count += 1
            checkpoint_to_drive(pending_files, f"dpo_sdxl_zsample_ckpt{ckpt_count}_seed{seed}")
            pending_files = []

    if pending_files:
        ckpt_count += 1
        checkpoint_to_drive(pending_files, f"dpo_sdxl_zsample_ckpt{ckpt_count}_seed{seed}")
        pending_files = []

print(f"\nDone. {done} images saved to {OUTPUT_DIR}")

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


[1/155] seed=2026 prompt=45 (34.1s)
[2/155] seed=2026 prompt=46 (34.2s)
[3/155] seed=2026 prompt=47 (34.2s)
[4/155] seed=2026 prompt=48 (34.1s)
[5/155] seed=2026 prompt=49 (34.2s)
[6/155] seed=2026 prompt=50 (34.2s)
[7/155] seed=2026 prompt=51 (34.2s)
[8/155] seed=2026 prompt=52 (34.2s)
[9/155] seed=2026 prompt=53 (34.2s)
[10/155] seed=2026 prompt=54 (34.1s)
[11/155] seed=2026 prompt=55 (34.1s)
[12/155] seed=2026 prompt=56 (34.1s)
[13/155] seed=2026 prompt=57 (34.0s)
[14/155] seed=2026 prompt=58 (34.3s)
[15/155] seed=2026 prompt=59 (34.0s)
  >> Checkpoint saved: dpo_sdxl_zsample_ckpt4_seed2026.zip (16.7 MB, 15 images)
[16/155] seed=2026 prompt=60 (34.3s)
[17/155] seed=2026 prompt=61 (34.0s)
[18/155] seed=2026 prompt=62 (34.0s)
[19/155] seed=2026 prompt=63 (33.8s)
[20/155] seed=2026 prompt=64 (33.8s)
[21/155] seed=2026 prompt=65 (34.0s)
[22/155] seed=2026 prompt=66 (34.1s)
[23/155] seed=2026 prompt=67 (34.2s)
[24/155] seed=2026 prompt=68 (33.8s)
[25/155] seed=2026 prompt=69 (33.8s)
[26/

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (88 > 77). Running this sequence through the model will result in indexing errors
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CLIPTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['rings on kthe end oc thei vatndlzes .']
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (88 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated bec

[66/155] seed=2026 prompt=110 (33.7s)
[67/155] seed=2026 prompt=111 (34.2s)


The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['can bce regulavewdd for lengh and vnolume y two or three pedalvs .']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['can bce regulavewdd for lengh and vnolume y two or three pedalvs .']


[68/155] seed=2026 prompt=112 (34.3s)
[69/155] seed=2026 prompt=113 (34.2s)
[70/155] seed=2026 prompt=114 (34.3s)
[71/155] seed=2026 prompt=115 (34.2s)
[72/155] seed=2026 prompt=116 (34.2s)
[73/155] seed=2026 prompt=117 (34.2s)
[74/155] seed=2026 prompt=118 (34.4s)
[75/155] seed=2026 prompt=119 (34.3s)
  >> Checkpoint saved: dpo_sdxl_zsample_ckpt8_seed2026.zip (20.9 MB, 15 images)
[76/155] seed=2026 prompt=120 (34.2s)
[77/155] seed=2026 prompt=121 (34.0s)
[78/155] seed=2026 prompt=122 (34.0s)
[79/155] seed=2026 prompt=123 (34.0s)
[80/155] seed=2026 prompt=124 (34.0s)
[81/155] seed=2026 prompt=125 (34.3s)
[82/155] seed=2026 prompt=126 (34.1s)
[83/155] seed=2026 prompt=127 (34.3s)
[84/155] seed=2026 prompt=128 (34.0s)
[85/155] seed=2026 prompt=129 (34.2s)
[86/155] seed=2026 prompt=130 (34.4s)
[87/155] seed=2026 prompt=131 (34.0s)
[88/155] seed=2026 prompt=132 (33.7s)
[89/155] seed=2026 prompt=133 (34.2s)
[90/155] seed=2026 prompt=134 (34.0s)
  >> Checkpoint saved: dpo_sdxl_zsample_ckpt9_

## Preview

In [ ]:
sample_paths = sorted(OUTPUT_DIR.rglob("*.png"))[-4:]
for p in sample_paths:
    print(p.name)
    display(Image.open(p))

## Copy to Google Drive

In [ ]:
for seed in SEEDS:
    seed_dir = OUTPUT_DIR / f"seed{seed}"
    if not seed_dir.exists():
        print(f"Skipping seed {seed} — no output dir")
        continue
    n = len(list(seed_dir.glob("*.png")))
    tag = f"dpo_sdxl_drawbench_zsampling_{n}imgs_seed{seed}_final"
    zp = shutil.make_archive(f"/content/{tag}", "zip", seed_dir)
    dest = DRIVE_DIR / f"{tag}.zip"
    shutil.copy2(zp, dest)
    print(f"{dest.name}: {Path(zp).stat().st_size / 1e6:.1f} MB")
os.sync()
print("All final zips flushed to Drive.")